In [ ]:
import pandas as pd

df = pd.read_csv("DataSet1-raw_listings.csv")

print(f"{df.shape[0]}/2040 listings and {df.shape[1]}/46 attributes loaded successfully.")

In [ ]:
print(df.head())

In [ ]:
print(df.isnull().sum())

In [ ]:
df = df.drop_duplicates(subset='vin')

In [ ]:
heading_fixes = {
    'Chevroletsilverado': '2004 Chevrolet Silverado',
    'Dodgestratus': 'Dodge Stratus',
    'Fordtaurus': 'Ford Taurus',
    'Hyundaisanta': 'Hyundai Santa Fe',
    'Lexussc': 'Lexus SC 430',
    'Pontiacvibe': 'Pontiac Vibe',
}

for wrong, correct in heading_fixes.items():
    mask = df['heading'].str.contains(wrong, case=False, na=False)
    df.loc[mask, 'heading'] = df.loc[mask, 'heading'].str.replace(wrong, correct, case=False, regex=False)

In [ ]:
df['heading'] = df['heading'].str.replace('MERCEDES-BENZ', 'Mercedes-Benz', regex=False)
df['heading'] = df['heading'].str.replace('Mercedes ', 'Mercedes-Benz ', regex=False)

In [ ]:
for wrong in heading_fixes.keys():
    remaining = df['heading'].str.contains(wrong, case=False, na=False).sum()
    print(f"{wrong}: {remaining} remaining")

In [ ]:
dash_fixes = {
    '-C-O-B-A-L-T-': 'Cobalt',
    '-G-R-A-N-D-': 'Grand Marquis',
    '-G-6-': 'G6',
    '-G-L-I-': 'Gli',
    '-E-2-5-0-': 'E-250',
    '-A-C-C-O-R-D-': 'Accord',
    '-Z-3-': 'Z3',
    '-Z-4-': 'Z4',
    '-C-3-0-': 'C30',
    '-V-5-0-': 'V50',
}

for wrong, correct in dash_fixes.items():
    df['heading'] = df['heading'].str.replace(wrong, correct, regex=False)

for wrong in dash_fixes.keys():
    remaining = df['heading'].str.contains(wrong, na=False, regex=False).sum()
    print(f"{wrong}: {remaining} remaining")

In [ ]:
no_year_mask = ~df['heading'].str.contains(r'(?:19|20)\d{2}', regex=True, na=True)
print(f"Dropping {no_year_mask.sum()} rows with no year in heading")
df = df[~no_year_mask]
print(f"Remaining rows: {len(df)}")

In [ ]:
import re

df['year'] = df['heading'].str.extract(r'((?:19|20)\d{2})')
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df = df.dropna(subset=['year'])
df['year'] = df['year'].astype(int)

df['make'] = df['heading'].str.extract(r'(?:19|20)\d{2}\s+(\w+)')
df['model'] = df['heading'].str.extract(r'(?:19|20)\d{2}\s+\w+\s+([\w-]+)')

df['make'] = df['make'].str.title().str.strip()
df['model'] = df['model'].str.title().str.strip()

df = df.dropna(subset=['make', 'model'])

print(f"Rows after extraction: {len(df)}")
print(df[['heading', 'year', 'make', 'model']].head(20))

In [ ]:
model_mapping = {
    'F150': 'F-150',
    'F250': 'F-250',
    'F350': 'F-350',
    'F-350Sd': 'F-350',
    'F450': 'F-450',
    'E150': 'E-150',
    
    'Crv': 'Cr-V',
    
    'Silverado-1500': 'Silverado',

    'Grand-Caravan': 'Grand Caravan',
    'Grand-Am': 'Grand Am',
    
    'Grand-Marquis': 'Grand Marquis',
    
    'Santa-Fe': 'Santa Fe',
    
    'Cx9': 'Cx-9',
    'Mazda3-Hatchback': 'Mazda3',
    '6': 'Mazda6',
    
    '93': '9-3',
    
    '5Series': '5-Series',
    
    'Xtype': 'X-Type',
    'Xj-Series': 'Xj',

    'Impreza-Outback-Sport': 'Impreza',
    
    'New': 'Beetle',
    
    'Es-350': 'Es',
    
    'Xl-7': 'Xl7',
}

df['model'] = df['model'].replace(model_mapping)

print(df['model'].value_counts())

In [ ]:
truncation_fixes = {
    ('Ford', 'Crown'): 'Crown Victoria',
    ('Ford', 'Five'): 'Five Hundred',
    ('Ford', 'Super'): 'Super Duty',
    ('Chrysler', 'Town'): 'Town And Country',
    ('Chrysler', 'Pt'): 'Pt Cruiser',
    ('Lincoln', 'Town'): 'Town Car',
    ('Lincoln', 'Mark'): 'Mark Viii',
    ('Mini', 'John'): 'John Cooper Works',
    ('Jeep', 'Grand'): 'Grand Cherokee',
    ('Dodge', 'Grand'): 'Grand Caravan',
    ('Pontiac', 'Grand'): 'Grand Am',
    ('Mercury', 'Grand'): 'Grand Marquis',
    ('Suzuki', 'Grand'): 'Grand Vitara',
    ('Land', 'Rover'): 'Range Rover',
}

for (make, wrong_model), correct_model in truncation_fixes.items():
    mask = (df['make'] == make) & (df['model'] == wrong_model)
    print(f"{make} {wrong_model}: {mask.sum()} rows → {correct_model}")
    df.loc[mask, 'model'] = correct_model

df.loc[df['make'] == 'Land', 'make'] = 'Land Rover'

In [ ]:
df.loc[df['make'] == 'Mercedes', 'make'] = 'Mercedes-Benz'

df.loc[df['make'] == 'Mazda3', 'model'] = 'Mazda3'
df.loc[df['make'] == 'Mazda3', 'make'] = 'Mazda'

df = df[df['make'] != 'Sterling']

df = df[df['make'] != 'Fordf']

print(f"Rows remaining: {len(df)}")
print(df['make'].value_counts())

In [ ]:
df['make'] = df['make'].str.title().str.strip()
df['model'] = df['model'].str.title().str.strip()

df = df.drop_duplicates(subset='vin')

# Final check
print(f"Final row count: {len(df)}")
print(f"Unique makes: {df['make'].nunique()}")
print(f"Unique models: {df['model'].nunique()}")
print(f"Unique make/model combinations: {df.groupby(['make', 'model']).ngroups}")
print(df[['make', 'model']].drop_duplicates().sort_values(['make', 'model']).to_string())

In [ ]:
df = df[df['make'] != '2004']

bmw_drop = ['3', '5', 'M', '7', '528', 'Mazda6']
df = df[~((df['make'] == 'Bmw') & (df['model'].isin(bmw_drop)))]

df = df[~((df['make'] == 'Chrysler') & (df['model'] == '300'))]

df = df[~((df['make'] == 'Infiniti') & (df['model'] == 'G'))]

jaguar_drop = ['Xj', 'Xj8', 'Xk', 'Xk8', 'Xkr', 'Xtype', 'Xf']
df = df[~((df['make'] == 'Jaguar') & (df['model'].isin(jaguar_drop)))]

df = df[~((df['make'] == 'Mazda') & (df['model'] == 'Mazda'))]

df = df[~((df['make'] == 'Hyundai') & (df['model'] == 'Santa'))]

print(f"Rows remaining: {len(df)}")
print(f"Unique make/model combinations: {df.groupby(['make', 'model']).ngroups}")

In [ ]:
df.loc[(df['make'] == 'Chevrolet') & (df['model'] == 'Monte'), 'model'] = 'Monte Carlo'
df.loc[(df['make'] == 'Saturn') & (df['model'] == 'Lseries'), 'model'] = 'L-Series'
df.loc[(df['make'] == 'Ford') & (df['model'] == 'E-350-Super-Duty'), 'model'] = 'E-350 Super Duty'

df = df[~((df['make'] == 'Infiniti') & (df['model'] == 'Qx'))]
df = df[~((df['make'] == 'Bmw') & (df['model'] == '530Xi'))]
df = df[~((df['make'] == 'Gmc') & (df['model'] == 'B-Series'))]

print(f"Rows remaining: {len(df)}")
print(f"Unique make/model combinations: {df.groupby(['make', 'model']).ngroups}")

df.to_csv("DataSet2-cleaned_listings.csv", index=False)
print("Saved DataSet2-cleaned_listings.csv")

**Wierdly Enough, MarketCheckAPI has Make and Model as params in the API documentation. How do they make it work if I needed all this normalization?**